# Genesis Studio on Kaggle (free GPU) + Cloudflare tunnel → Midox Vedios App

This notebook runs the **open-weights Genesis server** (Wan2.1-1.3B text→video, StableDiffusionXL image) on Kaggle's **free T4 x2 / P100**, then exposes it with a public Cloudflare tunnel URL that you paste into **Midox Vedios App → Settings → Genesis server URL**.

Setup (once):
1. Add the notebook → **Settings ▸ Accelerator → GPU (T4 x2)** and **Internet → On**.
2. Optional: Add-ons ▸ Secrets → `HF_TOKEN` (your Hugging Face token) to avoid download limits.
3. Run every cell top→bottom. The last cell prints `GENESIS_PUBLIC_URL = https://….trycloudflare.com` — that is the URL for the app.

Notes
- Weights are cached in `/kaggle/working/hf_cache` (persists between sessions, 20 GB cap) — later runs are much faster.
- GPU quota is ~30 h/week. Only enable the GPU accelerator when you actually use it.
- First video generation downloads Wan2.1-1.3B (~7 GB) → expect 25–40 min on a T4 (49 frames 832×480).
- Stack is pinned to `torch==2.6.0+cu124` on purpose: Kaggle's older Tesla P100 (sm_60) has no kernels in torch ≥ 2.7.

In [ ]:
# %% Cell 1 — GPU check + pinned install (torch 2.6.0 keeps P100 sm_60 + T4 sm_75 alive)
!nvidia-smi || echo '>>> NO GPU ACTIVE — check Settings ▸ Accelerator before running!'
%pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -q 'puter-ai-genesis @ git+https://github.com/mehedikhettabi3-dev/puter-ai-genesis.git'
%pip install -q diffusers transformers accelerate peft safetensors sentencepiece imageio imageio-ffmpeg 'uvicorn[standard]' fastapi pydantic pillow opencv-python-headless
print('install done')

In [ ]:
# %% Cell 2 — verify the stack imports (torch 2.6.0+cu124, genesis package, diffusers)
import torch, sys
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0), '| sm:', torch.cuda.get_device_capability(0))
import diffusers, transformers, accelerate
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__)
import genesis
from genesis.config import list_models
print('genesis', genesis.__version__, '| models:', [m['id'] for m in list_models()])
print('>>> stack OK')

In [ ]:
# %% Cell 3 — HF cache in /kaggle/working (persists across sessions) + optional HF_TOKEN secret
import os
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = tok
    print('HF_TOKEN: loaded from Kaggle secrets')
except Exception as e:
    print('HF_TOKEN: not set (anonymous downloads, may be rate limited) —', e)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# %% Cell 4 — fetch the FastAPI server (inference-server/) from the GitHub repo
import urllib.request, pathlib
BASE = 'https://raw.githubusercontent.com/mehedikhettabi3-dev/puter-ai-genesis/main/inference-server'
d = pathlib.Path('/kaggle/working/genesis-server'); d.mkdir(parents=True, exist_ok=True)
for name in ('server.py', 'jobs.py'):
    urllib.request.urlretrieve(f'{BASE}/{name}', d / name)
print('server files →', d)

In [ ]:
# %% Cell 5 — launch the Genesis server on :8500 (real diffusers engine, 0.0.0.0)
import sys, subprocess, time, urllib.request, json
d = '/kaggle/working/genesis-server'
srv = subprocess.Popen([sys.executable, 'server.py', '--host', '0.0.0.0', '--port', '8500'], cwd=d,
                       stdout=open(f'{d}/server.log', 'w'), stderr=subprocess.STDOUT)
ok = False
for _ in range(60):
    time.sleep(1)
    if srv.poll() is not None:
        print(srv.poll(), 'server exited — log tail:'); print(open(f'{d}/server.log').read()[-3000:]); break
    try:
        h = json.load(urllib.request.urlopen('http://127.0.0.1:8500/api/health', timeout=2))
        if h.get('ok'): ok = True; break
    except Exception:
        pass
print('health:', json.load(urllib.request.urlopen('http://127.0.0.1:8500/api/health')) if ok else 'FAILED')
print(json.dumps(json.load(urllib.request.urlopen('http://127.0.0.1:8500/api/models')), indent=1)[:1500])

In [ ]:
# %% Cell 6 — Cloudflare quick tunnel → public URL for Midox Vedios App
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared
import subprocess, time, re
open('/kaggle/working/tunnel.log', 'w').close()
tun = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8500'],
                       stdout=open('/kaggle/working/tunnel.log', 'ab'), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/kaggle/working/tunnel.log', 'rb').read().decode('utf-8', 'ignore'))
    if m: url = m.group(0); break
print(urllib.request.urlopen('http://127.0.0.1:8500/api/health').status, 'local ok?  tunnel:', tun.pid)
import urllib.request as u
h = u.urlopen(f'{url}/api/health', timeout=10).read().decode() if url else None
print('GENESIS_PUBLIC_URL =', url, '| remote health:', h)

In [ ]:
# %% Cell 7 — (optional) end-to-end test image generation — set True to burn a few minutes + DIN quota
RUN_TEST = False
import json, time, urllib.request as u
if RUN_TEST:
    job = json.loads(u.urlopen(u.Request('http://127.0.0.1:8500/api/generate',
        data=json.dumps({'kind': 'image', 'prompt': 'cinematic photo of a fox in the snow, 85mm', 'model': 'sd15',
                         'steps': 12, 'seed': 7}).encode(), headers={'Content-Type': 'application/json'}).open()))
    jid = job['job_id']
    for _ in range(3000):
        st = json.loads(u.urlopen(f'http://127.0.0.1:8500/api/jobs/{jid}').read())
        print(st['status'], st['progress']) if st['status'] != 'completed' else None
        if st['status'] in ('completed', 'failed', 'cancelled'): break
        time.sleep(1)
    print('final:', st)
    print('URL:', f'{url}/outputs/{st["result"]["file"]}' if st.get('result') else '—')

## Final step — connect the app

1. Copy the `GENESIS_PUBLIC_URL` from Cell 6 (e.g. `https://fast-word-xxxx.trycloudflare.com`).
2. Open **Midox Vedios App → Settings → Genesis server URL**, paste it, Save.
3. The models list now comes from the free Kaggle GPU server and generation runs there.

Keep the Kaggle tab open while using the app. Quit the GPU when done (right sidebar buttons) to save quota. Losing URL is normal: each session makes a new tunnel — just rerun Cells 1–6 and paste the new URL.